## 8-7. 문서 군집화 소개와 실습(Opinion Review 데이터 세트)

### 문서 군집화 개념

문서 군집화(Document Clustering）는 비슷한 텍스트 구성의 문서를 군집화（Clustering）하는 것입니 
다. 문서 군집화는 동일한 군집에 속하는 문서를 같은 카테고리 소속으로 분류할 수 있으므로 앞에서 
소개한 텍스트 분류 기반의 문서 분류와 유사합니다. 하지만 텍스트 분류 기반의 문서 분류는 사전에 
결정 카테고리 값을 가진 학습 데이터 세트가 필요한 데 반해, 문서 군집화는 학습 데이터 세트가 필요 
없는 비지도학습 기반으로 동작합니다. 이전 장에서 배웠던 군집화 기법을 활용해 텍스트 기반의 문서 
군집화를 적용하겠습니다

### Opinion Review 데이터 세트를 이용한 문서 군집화 수행하기

문서 군집화를 수행할 데이터 세트는 UCI 머신러닝 리포지토리에 있는 Opinion Review 데이터 세 
트입니다. 해당 데이터 세트는 51개의 텍스트 파일로 구성돼 있으며, 각 파일은 Tripadvisor（호텔）, Edmunds.com（자동차）, Amazon.com（전자제품） 사이트에서 가져온 리뷰 문서입니다. 각 문서는 약 
100개 정도의 문장을 가지고 있습니다. 원래는 토픽 모델링 논문으로 사용된 데이터 세트인데, 여기서는 문서 군집화를 이용해 각 리뷰를 분류해 보겠습니다.
전체 51개의 파일이 토요타와 같은 자동차 브랜드에 대한 평가와 아이팟 나노(ipod nano)의 음질과 같은 다양한 전자 제품과 호텔 서비스 등에 대한 리뷰 내용입니다. 이번 예제에서는 여러 개의 파일을 한 개의 DataFrame으로 로딩해 데이터 처리를 할 것입니다.


In [3]:
import pandas as pd
import glob, os 
import warnings 
warnings.filterwarnings('ignore') 
pd.set_option('display.max_colwidth', 700)

# 다음은 저자의 컴퓨터에서 압축 파일을 풀어놓은 디렉터리이니, 각자 디렉터리를 다시 설정합니다.
path = "C:/Users/User/OneDrive/바탕 화면/topics"
# path로 지정한 디렉터리 밑에 있는 모든 .data 파일들의 파일명을 리스트로 취합 
all_files = glob.glob(os.path.join(path, "*.data")) 
filename_list = [] 
opinion_text = []

# 개별 파일들의 파일명은 filename_list 리스트로 취합,
# 개별 파일들의 파일 내용은 DataFrame 로딩 후 다시 string으로 변환하여 opinion_text 리스트로 취합 
for file_ in all_files:
    # 개별 파일을 읽어서 DataFrame으로 생성
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
    # 절대경로로 주어진 파일명을 가공. Linux에서 수행 시에는 아래 \\를 / 변경.
    # 맨 마지막 .data 확장자도 제거
    filename_ = file_.split('\\')[-1]
    filename = filename_.split('.')[0]
    
    # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가.
    filename_list.append(filename)
    opinion_text.append(df.to_string())

# 파일명 리스트와 파일 내용 리스트를 DataFrame으로 생성
document_df = pd.DataFrame({'filename':filename_list, 'opinion_text': opinion_text}) 
document_df.head()

,filename,opinion_text
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi..."
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ..."
2,battery-life_amazon_kindle,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ..."
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now..."


각 파일 이름(filename) 자체만으로 의견(opinion)의 텍스트(text)가 어떠한 제품/서비스에 대한 리 
뷰인지 잘 알 수 있습니다.
문서를 TF—IDF 형태로 피처 벡터화하겠습니다. TfidfVectorizer는 Lemmatization 같은 어근 변환 
을 직접 지원하진 않지만, tokenizer 인자에 커스텀 어근 변환 함수를 적용해 어근 변환을 수행할 수 
있습니다. 이를 위해 LemNormalize() 함수를 만들겠습니다(해당 함수는 이전에 설명드린 단어 토큰화, 스톱 워드 제거, 어근 변환에 기반하고 있기에 여기서 언급 드리지 않고, 부록으로 제공되는 소스 
코드에서 보다 자세한 설명을 드리고 있으니, 참조 부탁드립니다). ngram은 (1,2)로 하고, min_df와 
max_df 범위를 설정해 피처의 개수를 제한하겠습니다. TfidfVectorizer의 fit_transform()의 인자로 
document df DataFrame의 opinion_text 칼럼을 입력하면 개별 문서 텍스트에 대해 TF-IDF 변환 
된 피처 벡터화된 행렬을 구할 수 있습니다.

In [9]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 필수 NLTK 데이터 다운로드(필요시)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

def LemNormalize(text):
  lemmatizer = WordNetLemmatizer()
  return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english', ngram_range=(1,2), min_df=0.05, max_df=0.85 )

#opinion_text 칼럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

문서별 텍스트가 TF-IDF 변환된 피처 벡터화 행렬 데이터에 대해서 군집화를 수행해 어떤 문서끼리 
군집되는지 확인해 보겠습니다. 군집화 기법은 K-평균을 적용하겠습니다. 문서의 유형은 크게 보자 
면, 전자제품, 자동차, 호텔로 돼 있습니다. 물론 전자 제품은 다시 내비게이션, 아이팟, 킨들, 랩톱 컴 
퓨터 등과 같은 세부 요소로 나뉩니다. 먼저 5개의 중심(Centroid) 기반으로 어떻게 군집화되는지 확 
인해 보겠습니다. 최대 반복 횟수 max_iter는 10000으로 설정합니다. KMeans를 수행한 후에 군집의 
Label 값과 중심별로 할당된 데이터 세트의 좌표 값을 구합니다.

In [ ]:
from sklearn.cluster import KMeans

# 5개 집합으로 군집화 수행. 예제를 위해 동일한 클러스터링 결과 도출용 random_state=0
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0) 
km_cluster.fit(feature_vect) 
cluster_label = km_cluster.labels_ 
cluster_centers = km_cluster.cluster_centers_

각 데이터별로 할당된 군집의 레이블을 파일명과 파일 내용을 가지고 있는 document_df DataFrame 
에 cluster_label 칼럼을 추가해 저장하겠습니다. 각 파일명은 의견 리뷰에 대한 주제를 나타냅니다. 
군집이 각 주제별로 유사한 형태로 잘 구성됐는지 알아보겠습니다.

In [14]:
document_df['cluster_label'] = cluster_label 
document_df.head()

,filename,opinion_text,cluster_label
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi...",2
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",1
2,battery-life_amazon_kindle,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",4
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,4
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",4


판다스 DataFrame의 sort_values（by=정렬칼럼명）를 수행하면 인자로 입력된 ‘정렬칼럼명’으로 데이 
터를 정렬할 수 있습니다. document_df DataFrame 객체에서 cluster_label로 어떤 파일명으로 매 
칭됐는지 보면서 군집화 결과를 확인해 보겠습니다. 먼저 cluster_label=0인 데이터 세트입니다.

Cluster#0 은 킨들, 아이팟, 넷북 등의 포터블 전자기기 및 주요 구성요소（배터리, 키보드등） 
에 대한 리뷰로 군집화돼 있습니다.

In [16]:
document_df[document_df['cluster_label']==0] .sort_values(by='filename')

,filename,opinion_text,cluster_label
5,buttons_amazon_kindle,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",0
10,eyesight-issues_amazon_kindle,"It feels as easy to read as the K1 but doesn't seem any crisper to my eyes .\n0 the white is really GREY, and to avoid considerable eye, strain I had to refresh pages every other page .\n1 The dream has always been a portable electronic device that could hold a ton of reading material, automate subscriptions and fa...",0
12,fonts_amazon_kindle,"Being able to change the font sizes is awesome !\n0 For whatever reason, Amazon decided to make the Font on the Home Screen ...",0
23,navigation_amazon_kindle,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",0
27,price_amazon_kindle,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",0
28,price_holiday_inn_london,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0


Cluster#1 은 호텔에 대한 리뷰로 군집화돼 있음을 알 수 있습니다

In [17]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",1
13,food_holiday_inn_london,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,1
14,food_swissotel_chicago,The food for our event was delicious .\n0 ...,1
15,free_bestwestern_hotel_sfo,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,1
20,location_bestwestern_hotel_sfo,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",1
21,location_holiday_inn_london,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",1
24,parking_bestwestern_hotel_sfo,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,1
32,room_holiday_inn_london,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",1
30,rooms_bestwestern_hotel_sfo,"Great Location , Nice Rooms , H...",1
31,rooms_swissotel_chicago,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",1


그런데 Cluster #2는 Cluster #0 과 비슷하게 킨들등이 군집에 포함돼 있지만, 주로 차량용내비게이션 
으로 군집이 구성돼 있음을 알 수 있습니다（파일명 XXX_garmin_nuvi_xxx는 차량용 내비게이션 리뷰 
입니다）

In [18]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n0 but for the most part, we find that the Garmin software provides accurate directions, whereever we intend to go .\n1 This functi...",2
8,directions_garmin_nuvi_255W_gps,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,2
9,display_garmin_nuvi_255W_gps,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",2
11,features_windows7,"I had to uninstall anti, virus and selected other programs, some of which did not have listings in the Programs and Features Control Panel section .\n0 This review briefly touches upon some of the key features and enhancements of Microsoft's latest OS .\n1 ...",2
19,keyboard_netbook_1005ha,", I think the new keyboard rivals the great hp mini keyboards .\n0 Since the battery life difference is minimum, the only reason to upgrade would be to get the better keyboard .\n1 The keyboard is now as good as t...",2
33,satellite_garmin_nuvi_255W_gps,"It's fast to acquire satellites .\n0 If you've ever had a Brand X GPS take you on some strange route that adds 20 minutes to your trip, has you turn the wrong way down a one way road, tell you to turn AFTER you've passed the street, frequently loses the satellite signal, or has old maps missing streets, you know how important this stuff is .\n1 ...",2
34,screen_garmin_nuvi_255W_gps,It is easy to read and when touching the screen it works great !\n0 and zoom out buttons on the 255w to the same side of the screen which makes it a bit easier .\n1 ...,2
35,screen_ipod_nano_8gb,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",2
36,screen_netbook_1005ha,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,2
41,size_asus_netbook_1005ha,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",2


Cluster #3는 토요타(Toyota)와 혼다(Honda) 등의 자동차에 대한 리뷰로 잘 군집화돼 있습니다

In [19]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
6,comfort_honda_accord_2008,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",3
7,comfort_toyota_camry_2007,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",3
16,gas_mileage_toyota_camry_2007,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,3
17,interior_honda_accord_2008,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,3
18,interior_toyota_camry_2007,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",3
22,mileage_honda_accord_2008,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",3
29,quality_toyota_camry_2007,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,3
37,seats_honda_accord_2008,"Front seats are very uncomfortable .\n0 No memory seats, no trip computer, can only display outside temp with trip odometer .\n1 ...",3
47,transmission_toyota_camry_2007,"After slowing down, transmission has to be kicked to speed up .\n0 ...",3


Cluster #4은 킨들(kindle) 리뷰가 한 개 섞여 있는 것이 살짝 아쉽지만, Cluster #1과 같이 대부분 호 
텔에 대한 리뷰로 군집화돼 있습니다.

In [35]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label


전반적으로 군집화된 결과를 살펴보면 군집 개수가 약간 많게 설정돼 있어서 세분화되어 군집화된 
경향이 있습니다. 중심 개수를 5개에서 3개로 낮춰서 3개 그룹으로 군집화한 뒤 결과를 확인해 보겠습 
니다.다음 소스 코드의 출력 결과가 길어서 개별 군집별로 요약된 결과만 책에 수록했습니다.

In [20]:
from sklearn.cluster import KMeans

# 3개의 집합으로 군집화
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

# 소속 클러스터를 cluster_label 칼럼으로 할당하고 cluster_label 값으로 정렬 
document_df ['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
25,performance_honda_accord_2008,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",0
23,navigation_amazon_kindle,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",0
12,fonts_amazon_kindle,"Being able to change the font sizes is awesome !\n0 For whatever reason, Amazon decided to make the Font on the Home Screen ...",0
10,eyesight-issues_amazon_kindle,"It feels as easy to read as the K1 but doesn't seem any crisper to my eyes .\n0 the white is really GREY, and to avoid considerable eye, strain I had to refresh pages every other page .\n1 The dream has always been a portable electronic device that could hold a ton of reading material, automate subscriptions and fa...",0
44,speed_windows7,"Windows 7 is quite simply faster, more stable, boots faster, goes to sleep faster, comes back from sleep faster, manages your files better and on top of that it's beautiful to look at and easy to use .\n0 , faster about 20% to 30% faster at running applications than my Vista , seriously\n1 ...",0
28,price_holiday_inn_london,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
27,price_amazon_kindle,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",0
5,buttons_amazon_kindle,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",0
29,quality_toyota_camry_2007,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,0
21,location_holiday_inn_london,"Great location for tube and we crammed in a fair amount of sightseeing in a short time .\n0 All in all, a normal chain hotel on a nice lo...",1


### 군집별 핵심 단어 추출하기

각 군집(Cluster)에 속한 문서는 핵심 단어를 주축으로 군집화돼 있을 것입니다. 이번에는 각 군집을 
구성하는 핵심 단어가 어떤 것이 있는지 확인해 보겠습니다. KMeans 객체는 각 군집을 구성하는 단어 피처가 군집의 중심(Centroid)을 기준으로 얼마나 가깝게 위치해 있는지 clusters_centers_라는 속성으로 제공합니다. clusters_centers_는 배열 값으로 제공되며, 행은 개별 군집을, 열은 개별 피처를 의미합니다. 각 배열 내의 값은 개별 군집 내의 상대 위치를 숫자 값으로 표현한 일종의 좌표 값입니다. 예를 들어 cluster_centers[O, 1]은 0번 군집에서 두 번째 피처의 위치 값입니다. 바로 앞 예제에서 군집 3개로 생성한 KMeans 객체인 km_cluster에서 cluster_centers, 속성값을 가져온 뒤 값을 확인해 보겠습니다.

In [21]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape:', cluster_centers.shape) 
print(cluster_centers)

cluster_centers shape: (3, 6155)
[[0.00146238 0.         0.         ... 0.         0.         0.        ]
 [0.         0.00029786 0.00065742 ... 0.00127666 0.0010113  0.0010113 ]
 [0.00138097 0.00149932 0.         ... 0.         0.         0.        ]]


cluster_centers_는 (3, 4611) 배열입니다. 이는 군집이 3개, word 피처가 4611개로 구성되었음을 의미합니다, 각 행의 배열 값은 각 군집 내의 4611개 피처의 위치가 개별 중심과 얼마나 가까운 가를 상대 값으로 나타낸 것입니다. 0에서 1까지의 값을 가질 수 있으며 1에 가까울수록 중심과 가까운 값을 의미합니다.이제 cluster_centers_ 속성값을 이용해 각 군집 별 핵심 단어를 찾아보겠습니다. cluster_centers_ 속성은 넘파이의 ndarray입니다. ndarray의 argsort( )[：,：：—1]를 이용하면 cluster_centers 배열 내 값이 큰 순으로 정렬된 위치 인덱스 값을 반환합니다(큰 값으로 정렬한 값을 반환하는 게 아니라 큰 값을 가진 배열 내 위치 인덱스 값을 반환하는 것입니다). 이 위치 인덱스 값이 필요한 이유는 핵심 단어 피처의 이름을 출력하기 위해서입니다. 새로운 함수 get_cluster_details()를 생성해 위에 대한 처리를 담당하겠습니다. cluster_centers_ 배열 내에서 가장 값이 큰 데이터의 위치 인덱스를 추출한 뒤, 해당 인덱스를 이용해 핵심 단어 이름과 그때의 상대 위치 값을 추출해 cluster_details라는 Dict 객체 변수에 기록하고 반환하는 것이 get_cluster_details() 함수의 주요 로직입니다.

In [22]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함.
def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10): 
    cluster_details = {}
    
    # cluster.centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환
    # 군집 중심점(centroid)별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함.
    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]
    
    # 개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력 
    for cluster_num in range(clusters_num):
        # 개별 군집별 정보를 담을 데이터 초기화.
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num
        
        # cluster_centers_.argsort()[：z ：：-1]로 구한 인덱스를 이용해 top n 피처 단어를 구함.
        top_feature_indexes = centroid_feature_ordered_ind[cluster_num,:top_n_features]
        top_features = [ feature_names[ind] for ind in top_feature_indexes ]
        
        # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값 구함.
        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()
        
        # cluster_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심위치 상댓값, 해당 파일명 입력 
        cluster_details[cluster_num] ['top_features'] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_feature_values 
        filenames = cluster_data[cluster_data['cluster_label'] == cluster_num] [ 'filename' ] 
        filenames = filenames.values.tolist()
        
        cluster_details[cluster_num]['filenames'] = filenames
    return cluster_details

get_cluster_details()를 호출하면 dictionary를 원소로 가지는 리스트인 cluster_delails를 반환합니다. 이 cluster_details에는 개별 군집번호, 핵심 단어, 핵심단어 중심 위치 상댓값, 파일명 속성 값 정보가 있는데, 이를 좀 더 보기 좋게 표현하기 위해서 별도의 print_cluster_details() 함수를 만들겠습니다.

In [30]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('####### Cluster {0}',format(cluster_num))
        print('Top features:', cluster_detail['top_features'])
        print('Reviews 파일명:', cluster_detail['filenames'][:7])
        print('===========================================')

이제 위에서 생성한 get_cluster_details( ), print_cluster_details( )를 호출해 보겠습니다. get_cluster_detail( ) 호출 시 인자는 KMeans 군집화 객체, 파일명 추출을 위한 document_df DataFrame, 핵심 단어 추출을 위한 피처명 리스트, 전체 군집 개수, 그리고 핵심 단어 추출 개수입니다. 피처명 리스트는 앞에서 TF-IDF 변환된 tfidflvect 객체에서 get_feature_names()로 추출하겠습니다.

In [31]:
feature_names = tfidf_vect.get_feature_names_out()
cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df,feature_names=feature_names, clusters_num=3, top_n_features=10 ) 
print_cluster_details(cluster_details)

####### Cluster {0} 0
Top features: ['price', 'kindle', 'page', 'button', 'font', 'book', 'performance', 'eye', 'quality', 'faster']
Reviews 파일명: ['buttons_amazon_kindle', 'eyesight-issues_amazon_kindle', 'fonts_amazon_kindle', 'navigation_amazon_kindle', 'performance_honda_accord_2008', 'price_amazon_kindle', 'price_holiday_inn_london']
####### Cluster {0} 1
Top features: ['room', 'service', 'hotel', 'staff', 'interior', 'food', 'location', 'seat', 'mileage', 'comfortable']
Reviews 파일명: ['bathroom_bestwestern_hotel_sfo', 'comfort_honda_accord_2008', 'comfort_toyota_camry_2007', 'food_holiday_inn_london', 'food_swissotel_chicago', 'free_bestwestern_hotel_sfo', 'gas_mileage_toyota_camry_2007']
####### Cluster {0} 2
Top features: ['screen', 'battery', 'battery life', 'life', 'keyboard', 'video', 'direction', 'voice', 'map', 'feature']
Reviews 파일명: ['accuracy_garmin_nuvi_255W_gps', 'battery-life_amazon_kindle', 'battery-life_ipod_nano_8gb', 'battery-life_netbook_1005ha', 'directions_garmi

포터블 전자제품 리뷰 군집인 Cluster #0에서는 screen’, ‘battery’, battery life’ 등과 같은 화면과 배터리 수명 등이 핵심 단어로 군집화되었습니다. 아무래도 모바일형이고 엔터테인먼트용 전자제품의 경우 화면 크기와 배터리 수명이 주요 관심사인 것 같습니다. 자동차 리뷰 군집인 Cluster #1에서는 ‘interior’, ‘seat’, ‘mileage’, ‘comfortable’ 등과 같은 실내 인테리어, 좌석, 연료 효율 등이 핵심 단어로 군집화되었습니다. 토요타, 혼다와 같은 일본 자동차의 경우 실내 인테리어와 연료 효율, 편안함이 주요 관심사로 보입니다. 호텔 리뷰 군집인 Cluster #2에서는 room’,‘hotel’, ‘service’, staff 등 같은 방과 서비스 등이 핵심 단어로 군집화되었습니다. 호텔 리뷰의 경우 방의 크기나 청소 상태, 직원들의 서비스, 위치 등이 주요 관심사입니다